# Pipeline Evaluation: EfficientNet\_B3 → PGA-UNet

Đánh giá pipeline 2 tầng trên tập `dataset_online` (ảnh có bệnh + ảnh bình thường).

## Cấu trúc dataset_online
```
dataset_online/
  images/          ← tất cả ảnh (.png), cả normal lẫn pathological
  annotations/     ← chỉ ảnh có bệnh mới có file .json
```

## Luồng pipeline
```
Tất cả ảnh → EfficientNet_B3 → Có bệnh? ──Có──→ PGA-UNet (zoom_out prompt) → Dice/IoU
                                          └─Không→ bỏ qua (không qua PGA)
```

## Cách tính 2 phần kết quả độc lập

### Phần 1 — Phân loại (EfficientNet_B3)
Đánh giá trên toàn bộ ảnh. GT suy từ JSON: có JSON = có bệnh, không có JSON = bình thường.

### Phần 2 — Phân đoạn pipeline (PGA-UNet)
Chỉ tính trên ảnh classifier gửi cho PGA (predicted positive = TP + FP).

| Trường hợp | JSON? | Dự đoán | PGA chạy? | Dice/IoU | Vào mẫu số? |
|---|---|---|---|---|---|
| TP | Có | Có bệnh ✅ | Có | Thực tế | ✅ |
| FP | Không | Có bệnh ❌ | Không (no JSON) | **0** | ✅ |
| FN | Có | Không bệnh ❌ | Không | — | ❌ không qua PGA |
| TN | Không | Không bệnh ✅ | Không | — | ❌ |

```
Pipeline_Dice = Σ Dice_i / N_predicted_positive   (TP→Dice thực, FP→0)
```
FP không có JSON nên không có GT mask → Dice=0 gán trực tiếp (không cần chạy PGA).  
FN không qua PGA → không tính vào metric phân đoạn.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
%cd /content
import os, torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Clone PGA-UNet source
if not os.path.exists('Prompt-Guided-XRay-Segmentation'):
    !git clone -b TN_B_ON https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git

!pip install -q gdown tqdm opencv-python scipy

# Download dataset_online
!gdown 1MZI6I0_aBxYhOFxhMCVNIPubcXtE7WLB -O dataset_online.zip -q
!unzip -oq dataset_online.zip

print('\n✅ Setup xong')

In [ ]:
# ── Cell 2: Download checkpoints ──────────────────────────────────────────────
import gdown

CLS_CKPT_ID = '1QuJdzR2l506H1uZ-4OzTdkKd7xZ0yk7A'
PGA_CKPT_ID = '1sSoQ8SObteWETuKSsGGhBASPtXQz9JbY'

os.makedirs('/content/checkpoints', exist_ok=True)
os.makedirs('/content/Prompt-Guided-XRay-Segmentation/checkpoints', exist_ok=True)

cls_ckpt = '/content/checkpoints/best_efficientnet_b3.pth'
pga_ckpt = '/content/Prompt-Guided-XRay-Segmentation/checkpoints/pga_unet_expB_best.pth'

if not os.path.exists(cls_ckpt):
    gdown.download(f'https://drive.google.com/uc?id={CLS_CKPT_ID}', cls_ckpt, quiet=False)
if not os.path.exists(pga_ckpt):
    gdown.download(f'https://drive.google.com/uc?id={PGA_CKPT_ID}', pga_ckpt, quiet=False)

assert os.path.exists(cls_ckpt), f'❌ Không tìm thấy: {cls_ckpt}'
assert os.path.exists(pga_ckpt), f'❌ Không tìm thấy: {pga_ckpt}'
print('✅ Checkpoints sẵn sàng')

In [ ]:
# ── Cell 3: Load models ────────────────────────────────────────────────────────
import sys, torch, torch.nn as nn
from torchvision import models as tv_models
sys.path.insert(0, '/content/Prompt-Guided-XRay-Segmentation')
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLS_THRESHOLD = 0.5    # ngưỡng phân loại (nhất quán với EfficientNet_B3.ipynb)
CLS_IMG_SIZE  = 300    # EfficientNet_B3
SEG_IMG_SIZE  = 512    # PGA-UNet

# --- EfficientNet_B3 ---
cls_model = tv_models.efficientnet_b3(weights=None)
cls_model.classifier[1] = nn.Linear(cls_model.classifier[1].in_features, 1)
cls_model.load_state_dict(torch.load(cls_ckpt, map_location=DEVICE, weights_only=True))
cls_model = cls_model.to(DEVICE).eval()
print('✅ EfficientNet_B3 loaded')

# --- PGA-UNet ---
seg_model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
seg_model.load_state_dict(torch.load(pga_ckpt, map_location=DEVICE, weights_only=True))
seg_model.eval()
print('✅ PGA-UNet loaded')

In [ ]:
# ── Cell 4: Helpers ────────────────────────────────────────────────────────────
import cv2, json, numpy as np
from PIL import Image
from torchvision import transforms

# --- Preprocessing ---

cls_transform = transforms.Compose([
    transforms.Resize((CLS_IMG_SIZE, CLS_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

seg_transform = transforms.Compose([
    transforms.Resize((SEG_IMG_SIZE, SEG_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def preprocess_cls(img_path):
    img = Image.open(img_path).convert('RGB')
    return cls_transform(img).unsqueeze(0).to(DEVICE)

def preprocess_seg(img_path):
    img = Image.open(img_path).convert('L')
    return seg_transform(img).unsqueeze(0).to(DEVICE)

# --- Per-polygon helpers ---

def load_polygons(json_path):
    """Đọc tất cả polygon từ annotation JSON (labelme format: shapes[i].points)."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return [s['points'] for s in data.get('shapes', [])
            if s.get('shape_type') == 'polygon']

def polygon_heatmap(polygon_pts, img_path, S=512, zoom_ratio=0.30):
    """Zoom-out Gaussian heatmap cho 1 polygon (khớp dataset.py test mode)."""
    orig = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    oh, ow = orig.shape[:2]
    pts = np.array(polygon_pts, dtype=np.float32)
    x_min, y_min = np.min(pts, axis=0)
    x_max, y_max = np.max(pts, axis=0)
    gt_w, gt_h = x_max - x_min, y_max - y_min
    bx_min = max(0,  x_min - gt_w * zoom_ratio)
    bx_max = min(ow, x_max + gt_w * zoom_ratio)
    by_min = max(0,  y_min - gt_h * zoom_ratio)
    by_max = min(oh, y_max + gt_h * zoom_ratio)
    # plateau heatmap in original space (matches create_plateau_heatmap)
    hm = np.zeros((oh, ow), dtype=np.float32)
    xi = max(0, int(bx_min - 5)); xa = min(ow, int(bx_max + 5))
    yi = max(0, int(by_min - 5)); ya = min(oh, int(by_max + 5))
    if xa > xi and ya > yi:
        hm[yi:ya, xi:xa] = 1.0
        hm = cv2.GaussianBlur(hm, (31, 31), 0)
    hm = cv2.resize(hm, (S, S))
    return torch.from_numpy(hm).unsqueeze(0).unsqueeze(0).to(DEVICE)

def polygon_gt_mask(polygon_pts, img_path, S=512):
    """GT binary mask cho 1 polygon (khớp dataset.py)."""
    orig = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    oh, ow = orig.shape[:2]
    pts = np.array(polygon_pts, dtype=np.float32)
    mask = np.zeros((oh, ow), dtype=np.uint8)
    cv2.fillPoly(mask, [pts.astype(np.int32)], 255)
    mask = cv2.resize(mask, (S, S), interpolation=cv2.INTER_NEAREST)
    return (mask > 127).astype(np.float32)

# --- Segmentation metrics (Dice + IoU) ---

def extract_lcc(m):
    if m.sum() == 0: return m
    n, lbl, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    return m if n <= 1 else (lbl == (1 + np.argmax(st[1:, cv2.CC_STAT_AREA]))).astype(np.float32)

def calc_dice_iou(pred_prob, gt_mask):
    pm  = extract_lcc((pred_prob > 0.5).astype(np.float32))
    gm  = (gt_mask > 0.5).astype(np.float32)
    eps = 1e-6
    tp  = (pm * gm).sum()
    fp  = (pm * (1 - gm)).sum()
    fn  = ((1 - pm) * gm).sum()
    dice = float((2*tp + eps) / (2*tp + fp + fn + eps))
    iou  = float((tp  + eps) / (tp  + fp + fn + eps))
    return dice, iou

print('✅ Helpers sẵn sàng')

In [ ]:
# ── Cell 5: Chạy pipeline trên dataset_online ──────────────────────────────────
from tqdm import tqdm

IMG_DIR  = '/content/dataset_online/images'
JSON_DIR = '/content/dataset_online/annotations'

all_images = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith('.png')])
json_set   = set(os.path.splitext(f)[0]
                 for f in os.listdir(JSON_DIR) if f.endswith('.json'))

N_total        = len(all_images)
N_pathological = len(json_set)
N_normal       = N_total - N_pathological

print(f'Tổng ảnh       : {N_total}')
print(f'Có bệnh (JSON) : {N_pathological}')
print(f'Bình thường    : {N_normal}')
print()

# results       : per-image, dùng cho classification metrics
# seg_samples   : per-polygon (TP) hoặc per-image (FP), dùng cho pipeline Dice/IoU
#   - TP polygon → dice/iou thực tế từ PGA
#   - FP image   → dice=0, iou=0 (không có GT mask)
results     = []
seg_samples = []

cls_model.eval()
seg_model.eval()

with torch.no_grad():
    for img_file in tqdm(all_images, desc='Pipeline'):
        base            = os.path.splitext(img_file)[0]
        img_path        = os.path.join(IMG_DIR,  img_file)
        jsn_path        = os.path.join(JSON_DIR, base + '.json')
        is_pathological = base in json_set

        # ── Tầng 1: Phân loại ──
        cls_prob   = torch.sigmoid(cls_model(preprocess_cls(img_path))).item()
        pred_label = 1 if cls_prob >= CLS_THRESHOLD else 0
        gt_label   = 1 if is_pathological else 0

        entry = {
            'sample'         : base,
            'gt_label'       : gt_label,
            'pred_label'     : pred_label,
            'cls_prob'       : cls_prob,
            'is_pathological': is_pathological,
        }

        if pred_label == 1:
            if is_pathological:
                # ── TP: tách từng polygon → mỗi polygon là 1 sample ──
                polygons = load_polygons(jsn_path)
                poly_dice_list, poly_iou_list = [], []
                for pts in polygons:
                    gt_mask = polygon_gt_mask(pts, img_path)
                    hm      = polygon_heatmap(pts, img_path)
                    prob_np = torch.sigmoid(
                        seg_model(preprocess_seg(img_path), hm)
                    )[0, 0].cpu().numpy()
                    d, u = calc_dice_iou(prob_np, gt_mask)
                    poly_dice_list.append(d)
                    poly_iou_list.append(u)
                    seg_samples.append({'seg_status': 'TP', 'dice': d, 'iou': u, 'img': base})
                entry.update({
                    'seg_status'  : 'TP',
                    'n_polygons'  : len(polygons),
                    'dice'        : float(np.mean(poly_dice_list)),
                    'iou'         : float(np.mean(poly_iou_list)),
                })
            else:
                # ── FP: không có GT → Dice=0, IoU=0 (1 đơn vị mẫu) ──
                seg_samples.append({'seg_status': 'FP', 'dice': 0.0, 'iou': 0.0, 'img': base})
                entry.update({'seg_status': 'FP', 'dice': 0.0, 'iou': 0.0, 'n_polygons': 0})
        else:
            if is_pathological:   # FN
                entry.update({'seg_status': 'FN', 'dice': None, 'iou': None, 'n_polygons': None})
            else:                 # TN
                entry.update({'seg_status': 'TN', 'dice': None, 'iou': None, 'n_polygons': 0})

        results.append(entry)

n_tp_samples = sum(1 for s in seg_samples if s['seg_status'] == 'TP')
n_fp_images  = sum(1 for s in seg_samples if s['seg_status'] == 'FP')
print(f'\n✅ Hoàn thành {len(results)} ảnh')
print(f'   TP samples (per-polygon) : {n_tp_samples}')
print(f'   FP images                : {n_fp_images}')
print(f'   Tổng đơn vị phân đoạn    : {n_tp_samples + n_fp_images}')

In [ ]:
# ── Cell 6: Kết quả ────────────────────────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score

# ── Đếm từng loại ảnh ──
n_tp_img = sum(1 for r in results if r['seg_status'] == 'TP')
n_fp_img = sum(1 for r in results if r['seg_status'] == 'FP')
n_fn_img = sum(1 for r in results if r['seg_status'] == 'FN')
n_tn_img = sum(1 for r in results if r['seg_status'] == 'TN')

# ── Đếm đơn vị phân đoạn ──
n_tp_samples = sum(1 for s in seg_samples if s['seg_status'] == 'TP')  # per-polygon
n_fp_images  = sum(1 for s in seg_samples if s['seg_status'] == 'FP')  # per-image (1 unit/img)
n_seg_total  = n_tp_samples + n_fp_images                               # mẫu số

# ════════════════════════════════════════════════════════════════════════
# BƯỚC 1 — PHÂN LOẠI (EfficientNet_B3): ai vào ai ra
# ════════════════════════════════════════════════════════════════════════
gt_labels   = [r['gt_label']   for r in results]
pred_labels = [r['pred_label'] for r in results]
cls_probs   = [r['cls_prob']   for r in results]

sensitivity = n_tp_img / (n_tp_img + n_fn_img) if (n_tp_img + n_fn_img) > 0 else 0.0
specificity = n_tn_img / (n_tn_img + n_fp_img) if (n_tn_img + n_fp_img) > 0 else 0.0
precision   = n_tp_img / (n_tp_img + n_fp_img) if (n_tp_img + n_fp_img) > 0 else 0.0
npv         = n_tn_img / (n_tn_img + n_fn_img) if (n_tn_img + n_fn_img) > 0 else 0.0
f1          = 2 * precision * sensitivity / (precision + sensitivity + 1e-9)
accuracy    = accuracy_score(gt_labels, pred_labels)
auc         = roc_auc_score(gt_labels, cls_probs)

print('═' * 65)
print('  BƯỚC 1 — PHÂN LOẠI (EfficientNet_B3)')
print('═' * 65)
print(f'  Input : {N_total} ảnh  ─  Có bệnh (GT): {N_pathological}  │  Bình thường (GT): {N_normal}')
print()
print(f'  Kết quả phân loại:')
print(f'    ✅ Có bệnh  → dự đoán Có bệnh   (TP) : {n_tp_img:>4} ảnh  ← đưa vào PGA')
print(f'    ❌ Có bệnh  → dự đoán Bình thường(FN) : {n_fn_img:>4} ảnh  ← bỏ sót, không qua PGA')
print(f'    ❌ Bình thường → dự đoán Có bệnh (FP) : {n_fp_img:>4} ảnh  ← đưa nhầm vào PGA → Dice=0')
print(f'    ✅ Bình thường → dự đoán Bình thường(TN):{n_tn_img:>4} ảnh  ← lọc đúng')
print()
print(f'  Metrics phân loại:')
print(f'    Sensitivity (Recall) : {sensitivity:.4f}  ─ {n_tp_img}/{n_tp_img+n_fn_img} ảnh bệnh phát hiện đúng')
print(f'    Specificity          : {specificity:.4f}  ─ {n_tn_img}/{n_tn_img+n_fp_img} ảnh bình thường lọc đúng')
print(f'    Precision            : {precision:.4f}')
print(f'    NPV                  : {npv:.4f}')
print(f'    F1-score             : {f1:.4f}')
print(f'    Accuracy             : {accuracy:.4f}')
print(f'    AUC-ROC              : {auc:.4f}')

# ════════════════════════════════════════════════════════════════════════
# BƯỚC 2 — ĐẦU VÀO PGA-UNet (predicted positive = TP + FP images)
# ════════════════════════════════════════════════════════════════════════
print()
print('═' * 65)
print('  BƯỚC 2 — ĐẦU VÀO PGA-UNet (ảnh được phân loại là Có bệnh)')
print('═' * 65)
n_pred_pos = n_tp_img + n_fp_img
print(f'  Tổng ảnh đưa vào PGA : {n_tp_img} (TP) + {n_fp_img} (FP) = {n_pred_pos} ảnh')
print()
print(f'  Phân tích {n_tp_img} ảnh TP (thực sự có bệnh):')
print(f'    → Tách polygon từ annotation JSON')
print(f'    → Tổng polygon (samples phân đoạn) : {n_tp_samples}')
if n_tp_img > 0:
    print(f'    → Trung bình polygon/ảnh           : {n_tp_samples/n_tp_img:.2f}')
print()
print(f'  Phân tích {n_fp_img} ảnh FP (bình thường bị đoán sai):')
print(f'    → Không có JSON (không có GT mask)')
print(f'    → Gán Dice=0, IoU=0 (mỗi ảnh = 1 đơn vị phân đoạn)')
print()
print(f'  Tổng đơn vị phân đoạn (mẫu số):')
print(f'    {n_tp_samples} polygon(TP) + {n_fp_images} ảnh(FP) = {n_seg_total} đơn vị')

# ════════════════════════════════════════════════════════════════════════
# BƯỚC 3 — KẾT QUẢ PHÂN ĐOẠN PIPELINE (PGA-UNet)
# ════════════════════════════════════════════════════════════════════════
tp_dice_sum = sum(s['dice'] for s in seg_samples if s['seg_status'] == 'TP')
tp_iou_sum  = sum(s['iou']  for s in seg_samples if s['seg_status'] == 'TP')

pipeline_dice = tp_dice_sum / n_seg_total if n_seg_total > 0 else 0.0
pipeline_iou  = tp_iou_sum  / n_seg_total if n_seg_total > 0 else 0.0

tp_dice_vals = [s['dice'] for s in seg_samples if s['seg_status'] == 'TP']

print()
print('═' * 65)
print('  BƯỚC 3 — KẾT QUẢ PHÂN ĐOẠN PIPELINE (PGA-UNet)')
print('═' * 65)
if tp_dice_vals:
    print(f'  Dice trung bình TP polygons : {np.mean(tp_dice_vals):.4f}  (chỉ tính ảnh bệnh đúng)')
print(f'  Σ Dice ({n_tp_samples} polygons)     : {tp_dice_sum:.4f}')
print(f'  Σ IoU  ({n_tp_samples} polygons)     : {tp_iou_sum:.4f}')
print()
print(f'  Pipeline Dice = {tp_dice_sum:.4f} / {n_seg_total} = {pipeline_dice:.4f}')
print(f'  Pipeline IoU  = {tp_iou_sum:.4f} / {n_seg_total} = {pipeline_iou:.4f}')

# ── Biểu đồ ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = np.linspace(0, 1, 21)
fp_dice_vals = [0.0] * n_fp_images

if tp_dice_vals:
    axes[0].hist(tp_dice_vals, bins=bins, color='steelblue', alpha=0.85,
                 label=f'TP: Dice thực tế ({n_tp_samples} polygons)')
if fp_dice_vals:
    axes[0].hist(fp_dice_vals, bins=bins, color='tomato', alpha=0.85,
                 label=f'FP: Dice=0 ({n_fp_images} ảnh)')
axes[0].axvline(pipeline_dice, color='black', linestyle='--',
                label=f'Pipeline Dice={pipeline_dice:.3f}')
axes[0].set_xlabel('Dice'); axes[0].set_ylabel('Số đơn vị')
axes[0].set_title(f'Phân bố Dice ({n_seg_total} đơn vị: {n_tp_samples} polygons + {n_fp_images} FP)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

cm_val = [[n_tn_img, n_fp_img], [n_fn_img, n_tp_img]]
cm_lbl = [['TN', 'FP'], ['FN', 'TP']]
axes[1].imshow(cm_val, cmap='Blues')
vmax = max(n_tn_img, n_fp_img, n_fn_img, n_tp_img)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{cm_val[i][j]}\n({cm_lbl[i][j]})',
                     ha='center', va='center', fontsize=13,
                     color='white' if cm_val[i][j] > vmax / 2 else 'black')
axes[1].set_xticks([0, 1]); axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Dự đoán: Bình thường', 'Dự đoán: Có bệnh'])
axes[1].set_yticklabels(['GT: Bình thường', 'GT: Có bệnh'])
axes[1].set_title('Confusion Matrix (Phân loại)')

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/pipeline_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: results/pipeline_evaluation.png')

In [ ]:
# ── Cell 7: Lưu CSV + Tóm tắt ─────────────────────────────────────────────────
import csv

# Lưu per-image results (classification)
csv_path = 'results/pipeline_detail.csv'
fieldnames = ['sample', 'gt_label', 'pred_label', 'cls_prob', 'seg_status',
              'n_polygons', 'dice', 'iou']
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for r in results:
        w.writerow({k: r.get(k, '') for k in fieldnames})

# Lưu per-polygon seg results
seg_csv_path = 'results/seg_samples.csv'
with open(seg_csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['img', 'seg_status', 'dice', 'iou'])
    w.writeheader()
    for s in seg_samples:
        w.writerow(s)

print(f'✅ CSV saved: {csv_path}')
print(f'✅ CSV saved: {seg_csv_path}')
print()
print('=' * 60)
print('  TÓM TẮT PIPELINE')
print('=' * 60)
print(f'  Tổng ảnh đầu vào  : {N_total}  (có bệnh: {N_pathological}, bình thường: {N_normal})')
print()
print(f'  [Phân loại — EfficientNet_B3]')
print(f'    TP={n_tp_img}  FP={n_fp_img}  FN={n_fn_img}  TN={n_tn_img}')
print(f'    Sensitivity : {sensitivity:.4f}  ({n_tp_img}/{n_tp_img+n_fn_img} ảnh bệnh phát hiện đúng)')
print(f'    Specificity : {specificity:.4f}  ({n_tn_img}/{n_tn_img+n_fp_img} ảnh bình thường loại đúng)')
print(f'    Accuracy    : {accuracy:.4f}')
print(f'    AUC-ROC     : {auc:.4f}')
print()
print(f'  [Pipeline phân đoạn — PGA-UNet]')
print(f'    TP samples (per-polygon) : {n_tp_samples}')
print(f'    FP images (Dice=0)       : {n_fp_images}')
print(f'    FN images (không tính)   : {n_fn_img}')
print(f'    Mẫu số (TP samples + FP) : {n_seg_total}')
print()
print(f'    Pipeline Dice = {pipeline_dice:.4f}   (Σ Dice_polygon / {n_seg_total})')
print(f'    Pipeline IoU  = {pipeline_iou:.4f}   (Σ IoU_polygon  / {n_seg_total})')